# Entrenamiento YOLO11n-seg — Segmentación de Pallets / Cargo
**Base model**: `yolo11n-seg.pt` (segmentación de instancia)  
**Output**: `pallet_seg.mlpackage` listo para iOS (Boxer3D app, reemplaza SAM)  

### Antes de correr
1. `Runtime → Change runtime type → T4 GPU`
2. Completá `ROBOFLOW_API_KEY`, `ROBOFLOW_WORKSPACE`, `ROBOFLOW_PROJECT` abajo
3. El dataset tiene que ser de **segmentación** (polígonos), no solo bounding boxes

In [ ]:
# ── Configuración ──────────────────────────────────────────────────────────
ROBOFLOW_API_KEY   = "gyo2JB6ByMr6sXmFn1lc"   # misma key que boxes
ROBOFLOW_WORKSPACE = ""   # ← completar: tu workspace de Roboflow
ROBOFLOW_PROJECT   = ""   # ← completar: nombre del proyecto de pallets
ROBOFLOW_VERSION   = 1

EPOCHS     = 100
IMGSZ      = 640
BATCH      = 16    # bajar a 8 si hay OOM en T4
BASE_MODEL = "yolo11n-seg.pt"

In [ ]:
# ── Verificar GPU ──────────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip() or 'NO GPU detectada')
assert r.returncode == 0, 'Necesitas GPU: Runtime → Change runtime type → T4'

In [ ]:
# ── Instalar dependencias ──────────────────────────────────────────────────
%pip install -q ultralytics roboflow coremltools

In [ ]:
# ── Descargar dataset de Roboflow ──────────────────────────────────────────
# El dataset tiene que estar en formato de SEGMENTACIÓN (polígonos).
# En Roboflow: Export → YOLOv8 (este formato funciona con YOLO11 seg también)
from roboflow import Roboflow

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download('yolov8')

DATA_YAML = dataset.location + '/data.yaml'
print('Dataset en:', dataset.location)

In [ ]:
# ── Inspeccionar clases del dataset ───────────────────────────────────────
import yaml

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

print('Clases:', cfg.get('names'))
print('nc:', cfg.get('nc'))
print('Task:', cfg.get('task', 'no especificado — asegurate de que sea segmentación'))
print()
print('--- data.yaml completo ---')
print(open(DATA_YAML).read())

# Guardar las clases para usarlas en el export a iOS
CLASS_NAMES = cfg.get('names', [])
print('\nEstas clases quedan en el modelo iOS:', CLASS_NAMES)

In [ ]:
# ── Verificar que las anotaciones son polígonos (segmentación) ─────────────
# Una anotación de detección tiene 5 valores por línea: clase x y w h
# Una anotación de segmentación tiene >5 valores:       clase x1 y1 x2 y2 ...
import glob, os

sample_labels = glob.glob(dataset.location + '/train/labels/*.txt')[:3]
for lf in sample_labels:
    with open(lf) as f:
        lines = f.readlines()
    if lines:
        n_vals = len(lines[0].strip().split())
        tipo = 'SEGMENTACIÓN ✓' if n_vals > 5 else 'DETECCIÓN (solo bbox, falta polígono)'
        print(f'{os.path.basename(lf)}: {n_vals} valores por línea → {tipo}')

In [ ]:
# ── Entrenar YOLO11n-seg ────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO(BASE_MODEL)

results = model.train(
    data          = DATA_YAML,
    epochs        = EPOCHS,
    imgsz         = IMGSZ,
    batch         = BATCH,
    patience      = 20,
    name          = 'pallet_seg',
    project       = '/content/runs',
    exist_ok      = True,
    optimizer     = 'AdamW',
    lr0           = 0.001,
    warmup_epochs = 3,
    # Augmentaciones: pallets se ven desde distintos ángulos
    mosaic    = 1.0,
    degrees   = 10.0,
    flipud    = 0.0,
    fliplr    = 0.5,
    hsv_h     = 0.015,
    hsv_s     = 0.7,
    hsv_v     = 0.4,
)

best_map_box  = results.results_dict.get('metrics/mAP50(B)', 0)
best_map_mask = results.results_dict.get('metrics/mAP50(M)', 0)
print(f'\n=== Entrenamiento finalizado ===')
print(f'mAP50 boxes: {best_map_box:.4f}')
print(f'mAP50 masks: {best_map_mask:.4f}   ← este es el que importa para segmentación')

In [ ]:
# ── Validar en test set ────────────────────────────────────────────────────
best_pt   = '/content/runs/pallet_seg/weights/best.pt'
model_val = YOLO(best_pt)
metrics   = model_val.val(data=DATA_YAML, split='test', imgsz=IMGSZ)

print('--- Boxes ---')
print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')
print()
print('--- Masks ---')
print(f'mAP50:     {metrics.seg.map50:.4f}')
print(f'mAP50-95:  {metrics.seg.map:.4f}')

In [ ]:
# ── Exportar a CoreML (.mlpackage) para iOS ────────────────────────────────
# nms=True: incluye NMS dentro del modelo (más simple en iOS)
# El modelo exportado reemplaza sam_encoder + sam_decoder en la app
model_val.export(
    format  = 'coreml',
    imgsz   = IMGSZ,
    nms     = True,
    simplify= True,
)

import os
coreml_path = best_pt.replace('.pt', '.mlpackage')
print(f'CoreML: {coreml_path}')
if os.path.exists(coreml_path):
    size_mb = sum(os.path.getsize(os.path.join(r,f))
                  for r,_,files in os.walk(coreml_path) for f in files) / 1e6
    print(f'Tamaño: {size_mb:.1f} MB')

In [ ]:
# ── Inspeccionar outputs del modelo CoreML ─────────────────────────────────
# Necesitamos saber los nombres de los outputs para parsearlos en Swift
import coremltools as ct

cml = ct.models.MLModel(coreml_path)
spec = cml.get_spec()

print('=== INPUTS ===')
for inp in spec.description.input:
    shape = list(inp.type.multiArrayType.shape)
    print(f'  {inp.name}: {shape}')

print()
print('=== OUTPUTS ===')
for out in spec.description.output:
    shape = list(out.type.multiArrayType.shape)
    print(f'  {out.name}: {shape}')

print()
print('Guardá estos nombres — los necesitás para PalletDetector.swift en iOS')

In [ ]:
# ── Comprimir y descargar ──────────────────────────────────────────────────
import shutil
from google.colab import files

zip_path = '/content/pallet_seg'
shutil.make_archive(zip_path, 'zip', coreml_path)
files.download(zip_path + '.zip')

print('Descargado: pallet_seg.zip')
print()
print('Próximos pasos en Xcode:')
print('  1. Descomprimir → pallet_seg.mlpackage')
print('  2. Arrastrar al proyecto boxer/ (mismo lugar que sam_encoder/sam_decoder)')
print('  3. En ARViewModel: reemplazar SAMSegmenter por PalletDetector (nuevo archivo)')
print(f'  4. Clases del modelo: {CLASS_NAMES}')